In [ ]:
!pip install cellpose

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
from cellpose import models, io
import cv2
import torch

# ============ LOAD CELLPOSE ============
model = models.CellposeModel(gpu=True, model_type='cyto2')


# Verify GPU usage
if torch.cuda.is_available():
    model.net.to(torch.device("cuda"))
    print(f"✅ Model on GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ GPU not available, using CPU")

print(f"Model device: {next(model.net.parameters()).device}")

# ============ PATHS ============
noisy_folder = Path("data/microscopy/png_slices")
n2d_folder = Path("results/microscopy/N2D")
n2d_folder_colie = Path("results/microscopy/Sequential_N2D_COLIE")
zs_pipeline = Path("results/microscopy/zsn2n_pipeline")
n2d_pipeline = Path("results/microscopy/n2d_pipeline_250_1000")
output_dir = Path("results/downstream_tasks/microscopy_bioluminescence")
output_dir.mkdir(parents=True, exist_ok=True)

csv_path = output_dir / "cellpose_comparison.csv"

def segment_cellpose(image_path):
    image = io.imread(str(image_path))
    if image.ndim == 2:
        image = np.stack([image, image, image], axis=2)
    if image.max() > 1:
        image = image.astype(np.float32) / 255.0
    masks, flows, styles = model.eval(image, diameter=None, channels=[0, 0], do_3D=False,normalize=False) #normalize=False to avoid changing the image intensity
    return masks, image

def create_overlay_cellpose(image, masks):
    result = image.copy()
    if masks is not None and len(np.unique(masks)) > 1:
        result = result.astype(np.float32)
        unique_masks = np.unique(masks)[1:]
        colors = plt.cm.tab20(np.linspace(0, 1, len(unique_masks)))
        for i, mask_id in enumerate(unique_masks):
            color = colors[i % len(colors)]
            mask = (masks == mask_id).astype(np.float32)
            for c in range(3):
                result[:, :, c] = result[:, :, c] * (1 - mask) + color[c] * mask * 0.5
        result = np.clip(result, 0, 1)
    return result

def count_cells(masks):
    return len(np.unique(masks)) - 1

def compare_pipelines(noisy_folder, folders, output_dir, n_images=None):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    noisy_files = sorted(noisy_folder.glob("*.png"))
    if n_images is not None:
        noisy_files = noisy_files[:n_images]
    
    csv_path = output_dir / "cellpose_comparison.csv"
    
    # Load existing results
    results = []
    completed = set()
    if csv_path.exists():
        try:
            existing_df = pd.read_csv(csv_path)
            completed = set(existing_df['filename'].tolist())
            results = existing_df.to_dict('records')
            print(f"✅ Found {len(completed)} already processed images")
        except Exception as e:
            print(f"⚠️ Could not read existing CSV: {e}")
    
    # Filter out already processed files
    remaining_files = [f for f in noisy_files if f.name not in completed]
    
    if len(remaining_files) == 0:
        print("🎉 All images already processed!")
        return results
    
    print(f"📊 Processing {len(remaining_files)} remaining images...")
    
    method_names = list(folders.keys())
    
    for img_path in remaining_files:
        img_name = img_path.name
        print(f"\n--- {img_name} ---")
        
        img_results = {'filename': img_name}
        overlays = {}
        originals = {}
        cell_counts = {}
        
        for method_name, folder_path in folders.items():
            method_path = folder_path / img_name
            if not method_path.exists():
                print(f"  ⚠️ {method_name}: image not found at {method_path}")
                continue
            
            try:
                masks, img = segment_cellpose(method_path)
                num_cells = count_cells(masks)
                cell_counts[method_name] = num_cells
                img_results[f'num_cells_{method_name}'] = num_cells
                originals[method_name] = img
                overlays[method_name] = create_overlay_cellpose(img, masks)
                print(f"  ✅ {method_name}: {num_cells} cells")
            except Exception as e:
                print(f"  ❌ {method_name}: Error - {e}")
                img_results[f'num_cells_{method_name}'] = None
        
        # Create comparison grid
        n_methods = len(folders)
        fig, axes = plt.subplots(n_methods, 2, figsize=(12, n_methods * 4))
        if n_methods == 1:
            axes = axes.reshape(1, -1)
        
        for idx, (method_name, _) in enumerate(folders.items()):
            if method_name in originals:
                axes[idx, 0].imshow(originals[method_name])
                axes[idx, 0].set_title(f"{method_name} (Original)", fontsize=11)
                axes[idx, 0].axis('off')
                axes[idx, 1].imshow(overlays[method_name])
                axes[idx, 1].set_title(f"{method_name} - {cell_counts.get(method_name, 0)} cells", fontsize=11)
                axes[idx, 1].axis('off')
            else:
                axes[idx, 0].axis('off')
                axes[idx, 1].axis('off')
        
        fig.suptitle(img_name, fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.savefig(output_dir / f"{img_name}", dpi=150, bbox_inches='tight')
        plt.close()
        
        # Add to results
        results.append(img_results)
        
        # --- SAVE CSV IMMEDIATELY AFTER EACH IMAGE ---
        df = pd.DataFrame(results)
        # Ensure all columns exist
        for method in method_names:
            if f'num_cells_{method}' not in df.columns:
                df[f'num_cells_{method}'] = None
        df.to_csv(csv_path, index=False)
        print(f"  💾 CSV saved ({len(results)} images total)")
    
    return results

# ============ RUN ============
folders = {
    'Noisy': noisy_folder,
    'N2D': n2d_folder,
    'N2D_Colie': n2d_folder_colie,
    'ZS_Pipeline': zs_pipeline,
    'N2D_Pipeline': n2d_pipeline,
}

results = compare_pipelines(
    noisy_folder=noisy_folder,
    folders=folders,
    output_dir=output_dir,
    n_images=None
)

# ============ SUMMARY ============
if results:
    print("\n" + "="*80)
    print("CELLPOSE SUMMARY")
    print("="*80)
    method_names = list(folders.keys())
    print(f"{'Filename':<35}", end="")
    for method in method_names:
        print(f"{method:<15}", end="")
    print()
    print("-" * (35 + 15 * len(method_names)))
    for r in results:
        print(f"{r['filename']:<35}", end="")
        for method in method_names:
            val = r.get(f'num_cells_{method}', 'N/A')
            print(f"{val if val is not None else 'N/A':<15}", end="")
        print()
    
    print("-" * (35 + 15 * len(method_names)))
    print(f"{'AVERAGE':<35}", end="")
    for method in method_names:
        vals = [r.get(f'num_cells_{method}', 0) for r in results if r.get(f'num_cells_{method}') is not None]
        avg = np.mean(vals) if vals else 0
        print(f"{avg:<15.2f}", end="")
    print()

print(f"\n✅ Results saved to: {csv_path}")